# 📊 Exploratory Data Analysis — Smart Loan Recovery
**Phase 1 | 2026-04-27**
**Dataset:** `credit_risk_dataset.csv` (32,581 records × 12 features)

**Source files:**
- `C:/Users/dell/Documents/GitHub/smart-loan-recovery/data/raw/accepted_2007_to_2018Q4.csv` *(LendingClub accepted loans — not used in initial EDA due to size)*
- `C:/Users/dell/Documents/GitHub/smart-loan-recovery/data/raw/rejected_2007_to_2018Q4.csv` *(LendingClub rejected loans — permission issue, deferred)*
- `C:/Users/dell/Documents/GitHub/smart-loan-recovery/data/raw/credit_risk_dataset.csv` ✅ **Active dataset**
- `C:/Users/dell/Documents/GitHub/smart-loan-recovery/data/raw/Data Dictionary.xls`

## 1. Data Quality Check

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
DATA_PATH = 'C:/Users/dell/Documents/GitHub/smart-loan-recovery/data/raw/credit_risk_dataset.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing': missing, 'Pct': missing_pct}))

In [ ]:
df.describe().round(2)

In [ ]:
from IPython.display import display, Image
display(Image('../notebooks/01_quality.png'))

## 2. Target Distribution
**Target:** `loan_status` — 0 = Non-Default, 1 = Default

In [ ]:
counts = df['loan_status'].value_counts().sort_index()
print('Class counts:')
print(counts)
print('\nClass proportions:')
print(df['loan_status'].value_counts(normalize=True).round(4) * 100)

In [ ]:
display(Image('../notebooks/02_target_dist.png'))

## 3. Categorical Feature Distributions

In [ ]:
for col in ['person_home_ownership','loan_intent','loan_grade','cb_person_default_on_file']:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()

In [ ]:
display(Image('../notebooks/04_cat_dist.png'))

## 4. Correlation Matrix

In [ ]:
corr_cols = ['person_age','person_income','person_emp_length','loan_amnt',
             'loan_int_rate','loan_percent_income','cb_person_cred_hist_length','loan_status']
corr = df[corr_cols].corr()
print('Correlation with target (loan_status):')
print(corr['loan_status'].drop('loan_status').sort_values(ascending=False).round(4))

In [ ]:
display(Image('../notebooks/05_corr_matrix.png'))

## 5. Feature Distributions & Key Predictors

In [ ]:
# Numerical feature distributions
display(Image('../notebooks/03_num_dist.png'))
# Boxplots: key predictors vs default
display(Image('../notebooks/06_boxplots.png'))

In [ ]:
display(Image('../notebooks/07_grade_default.png'))

## 6. EDA Summary

### Dataset Overview
- **Source:** `credit_risk_dataset.csv` — 32,581 records × 12 features
- **Target:** `loan_status` — binary (0=Non-Default, 1=Default)
- **Additional data (deferred):** LendingClub accepted/rejected files (too large for initial EDA; to be merged later)

### Data Quality
- **No missing target values**
- **2 columns with missing data:** `loan_int_rate` (9.6%), `person_emp_length` (2.7%)
- No duplicate rows detected
- `person_age` has unrealistic outliers (max=144) — recommend filtering age > 100
- **Categorical features:** `person_home_ownership`, `loan_intent`, `loan_grade`, `cb_person_default_on_file` — all clean

### Target Distribution (Class Imbalance)
| Class | Count | Proportion |
|---|---|---|
| Non-Default (0) | 25,473 | 78.2% |
| Default (1) | 7,108 | 21.8% |

**~4:1 imbalance** — significant but not extreme. Class weights or SMOTE recommended for modeling.

### Strongest Predictors of Default (by correlation with `loan_status`)
| Rank | Feature | Correlation | Direction |
|---|---|---|---|
| 1 | `loan_int_rate` | **+0.38** | Higher rate → higher default |
| 2 | `loan_percent_income` | **+0.32** | Larger loan vs income → higher risk |
| 3 | `cb_person_default_on_file` | encoded separately | Prior default strongly predictive |
| 4 | `loan_grade` | A→G, higher= worse | Ordinal encoding needed |
| 5 | `person_income` | **−0.19** | Higher income → lower default |

### Key Insights
1. **Interest rate is the single strongest predictor** of default — aligns with lending domain knowledge
2. **Loan percent of income** is a strong secondary signal — borrowers taking large loans relative to income are high-risk
3. **Prior default on file** (`cb_person_default_on_file`) is a critical binary flag
4. **Loan grade** (A–G) is an ordinal risk tier — subgrade interactions worth exploring
5. **Age** and **credit history length** show mild negative correlation with default (older/more experienced borrowers are safer)

### Recommended Next Steps (Phase 2)
1. Filter `person_age > 100` as data errors
2. Impute `loan_int_rate` (median or model-based) and `person_emp_length`
3. Ordinal-encode `loan_grade`; binary-encode `cb_person_default_on_file`
4. One-hot encode `loan_intent`, `person_home_ownership`
5. Handle class imbalance: `class_weight='balanced'` or SMOTE on training set
6. Build baseline models: Logistic Regression, Random Forest, XGBoost
7. Evaluate with AUC-ROC, F1, Precision-Recall curves